# PNEUNET_MR PLOT

In [9]:
# %% RELOAD bundle and re-plot (save PDF + PNG to same subfolder)
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ---- Fonts (Times-like) + avoid Type 3) ----
mpl.rcParams.update({
    'font.family': ['Times New Roman', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'text.usetex': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

# ===== SETTINGS =====
PARENT   = "save_plots"
SUB      = "PNEUNET_MR"
BUNDLE   = "PNEUNET_MR_snap25.npz"   # change if you saved a different name
PNG_DPI  = 1440
# ====================

path = os.path.join(PARENT, SUB, BUNDLE)
if not os.path.exists(path):
    raise FileNotFoundError("Bundle not found: " + os.path.abspath(path))

Z = np.load(path, allow_pickle=True)

# unpack
SNAP   = int(Z["snap"])
x_ic   = Z["x_ic"];  y_ic = Z["y_ic"]
u_star = Z["u_star"]; v_star = Z["v_star"]
u_pred = Z["u_pred"]; v_pred = Z["v_pred"]
XLIM   = tuple(Z["xlim"])
CMAP   = str(Z["cmap"])

label_ux     = Z["label_ux"].item()
label_ux_hat = Z["label_ux_hat"].item()
label_uy     = Z["label_uy"].item()
label_uy_hat = Z["label_uy_hat"].item()

umin_u = float(Z["umin_u"]); umax_u = float(Z["umax_u"])
vmin_v = float(Z["vmin_v"]); vmax_v = float(Z["vmax_v"])

norm_u = Normalize(vmin=umin_u, vmax=umax_u)
norm_v = Normalize(vmin=vmin_v, vmax=vmax_v)

def plot_deformation(
    x, y, u, v, comp_label, norm, component='u',
    cmap='jet', xlim=None, textbox_xy=(0.035, 0.05),
    title=None, save_stub=None,
    # --- font controls ---
    cbar_label_fs=28, cbar_tick_fs=18, box_fs=21,
    label_fs=28, tick_fs=18, title_fs=20,
):
    x = x.ravel(); y = y.ravel()
    u = u.ravel(); v = v.ravel()

    # sign convention
    x_def = -x - u
    y_def =  y + v

    if component == 'u':
        cfield = u
        label_text = f"Max {comp_label}: {u.max():.2f}\nMin {comp_label}: {u.min():.2f}"
    else:
        cfield = v
        label_text = f"Max {comp_label}: {v.max():.2f}\nMin {comp_label}: {v.min():.2f}"

    fig, ax = plt.subplots(figsize=(10, 6), dpi=1440, constrained_layout=False)

    # --- FIX: keep a constant margin so big labels don't get cut ---
    # these fractions are figure-relative and identical across all plots
    fig.subplots_adjust(left=0.14, right=0.90, bottom=0.22, top=0.95)

    # plots
    ax.scatter(-x, y, color='grey', alpha=1, s=0.009)                       # undeformed
    ax.scatter(x_def, y_def, c=cfield, cmap=cmap, norm=norm, s=1, alpha=1)  # deformed

    # colorbar (attached to ax; margins above leave consistent space)
    cbar = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax,
                        fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=cbar_tick_fs)

    # Stats box
    ax.text(0.03, 0.03, label_text, transform=ax.transAxes,
            fontsize=box_fs, va='bottom',linespacing=0.95,
            bbox=dict(boxstyle='round,pad=0.18,rounding_size=0.06', facecolor='white', alpha=0.4))

    # labels / ticks / aspect
    ax.set_xlabel('X Coordinate', fontsize=label_fs, labelpad=8)  # <-- pad
    ax.set_ylabel('Y Coordinate', fontsize=label_fs, labelpad=6)  # <-- pad
    ax.tick_params(axis='both', labelsize=tick_fs)
    ax.set_aspect('equal', adjustable='datalim')  # consistent box; adjust limits

    if xlim is not None:
        ax.set_xlim(*xlim)
    if title:
        ax.set_title(title, fontsize=title_fs, pad=6)

    # Save without tight cropping
    if save_stub:
        base = os.path.join(PARENT, SUB, save_stub)
        fig.savefig(base + ".pdf")
        fig.savefig(base + ".png", dpi=PNG_DPI)

    plt.close(fig)




# Reproduce the two plots (u_y GT and Prediction). Add u_x if needed.

plot_deformation(x_ic, y_ic, u_star, v_star, comp_label=label_ux, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=None, save_stub=f"PNEUNET_MR_ux_EXACT_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)
plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_ux_hat, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=None, save_stub=f"PNEUNET_MR_ux_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

plot_deformation(
    x_ic, y_ic, u_star, v_star,
    comp_label=label_uy, norm=norm_v, component='v', cmap=CMAP,
    xlim=XLIM, title=None, save_stub=f"PNEUNET_MR_uy_EXACT_snap{SNAP}_reload",
    cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_uy_hat, norm=norm_v,
                 component='v', cmap=CMAP, xlim=XLIM, title=None, save_stub=f"PNEUNET_MR_uy_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)



# PNEUNET_YEOH PLOT

In [10]:
# %% RELOAD bundle and re-plot (save PDF + PNG to same subfolder)
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ---- Fonts (Times-like) + avoid Type 3) ----
mpl.rcParams.update({
    'font.family': ['Times New Roman', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'text.usetex': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

# ===== SETTINGS =====
PARENT   = "save_plots"
SUB      = "PNEUNET_YEOH"
BUNDLE   = "PNEUNET_YEOH_snap25.npz"   # change if you saved a different name
PNG_DPI  = 1440
# ====================

path = os.path.join(PARENT, SUB, BUNDLE)
if not os.path.exists(path):
    raise FileNotFoundError("Bundle not found: " + os.path.abspath(path))

Z = np.load(path, allow_pickle=True)

# unpack
SNAP   = int(Z["snap"])
x_ic   = Z["x_ic"];  y_ic = Z["y_ic"]
u_star = Z["u_star"]; v_star = Z["v_star"]
u_pred = Z["u_pred"]; v_pred = Z["v_pred"]
XLIM   = tuple(Z["xlim"])
CMAP   = str(Z["cmap"])

label_ux     = Z["label_ux"].item()
label_ux_hat = Z["label_ux_hat"].item()
label_uy     = Z["label_uy"].item()
label_uy_hat = Z["label_uy_hat"].item()

umin_u = float(Z["umin_u"]); umax_u = float(Z["umax_u"])
vmin_v = float(Z["vmin_v"]); vmax_v = float(Z["vmax_v"])

norm_u = Normalize(vmin=umin_u, vmax=umax_u)
norm_v = Normalize(vmin=vmin_v, vmax=vmax_v)

def plot_deformation(
    x, y, u, v, comp_label, norm, component='u',
    cmap='jet', xlim=None, textbox_xy=(0.035, 0.05),
    title=None, save_stub=None,
    # --- font controls ---
    cbar_label_fs=28, cbar_tick_fs=18, box_fs=21,
    label_fs=28, tick_fs=18, title_fs=20,
):
    x = x.ravel(); y = y.ravel()
    u = u.ravel(); v = v.ravel()

    # sign convention
    x_def = -x - u
    y_def =  y + v

    if component == 'u':
        cfield = u
        label_text = f"Max {comp_label}: {u.max():.2f}\nMin {comp_label}: {u.min():.2f}"
    else:
        cfield = v
        label_text = f"Max {comp_label}: {v.max():.2f}\nMin {comp_label}: {v.min():.2f}"

    fig, ax = plt.subplots(figsize=(10, 6), dpi=1440, constrained_layout=False)

    # --- FIX: keep a constant margin so big labels don't get cut ---
    # these fractions are figure-relative and identical across all plots
    fig.subplots_adjust(left=0.14, right=0.90, bottom=0.22, top=0.95)

    # plots
    ax.scatter(-x, y, color='grey', alpha=1, s=0.009)                       # undeformed
    ax.scatter(x_def, y_def, c=cfield, cmap=cmap, norm=norm, s=1, alpha=1)  # deformed

    # colorbar (attached to ax; margins above leave consistent space)
    cbar = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax,
                        fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=cbar_tick_fs)

    # Stats box
    ax.text(0.03, 0.03, label_text, transform=ax.transAxes,
            fontsize=box_fs, va='bottom',linespacing=0.95,
            bbox=dict(boxstyle='round,pad=0.18,rounding_size=0.06', facecolor='white', alpha=0.4))

    # labels / ticks / aspect
    ax.set_xlabel('X Coordinate', fontsize=label_fs, labelpad=8)  # <-- pad
    ax.set_ylabel('Y Coordinate', fontsize=label_fs, labelpad=6)  # <-- pad
    ax.tick_params(axis='both', labelsize=tick_fs)
    ax.set_aspect('equal', adjustable='datalim')  # consistent box; adjust limits

    if xlim is not None:
        ax.set_xlim(*xlim)
    if title:
        ax.set_title(title, fontsize=title_fs, pad=6)

    # Save without tight cropping
    if save_stub:
        base = os.path.join(PARENT, SUB, save_stub)
        fig.savefig(base + ".pdf")
        fig.savefig(base + ".png", dpi=PNG_DPI)

    plt.close(fig)


# Reproduce the two plots (u_y GT and Prediction). Add u_x if needed.

plot_deformation(x_ic, y_ic, u_star, v_star, comp_label=label_ux, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=None, save_stub=f"PNEUNET_YEOH_ux_EXACT_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)
plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_ux_hat, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=None, save_stub=f"PNEUNET_YEOH_ux_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

plot_deformation(
    x_ic, y_ic, u_star, v_star,
    comp_label=label_uy, norm=norm_v, component='v', cmap=CMAP,
    xlim=XLIM, title=None, save_stub=f"PNEUNET_YEOH_uy_EXACT_snap{SNAP}_reload",
    cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_uy_hat, norm=norm_v,
                 component='v', cmap=CMAP, xlim=XLIM, title=None, save_stub=f"PNEUNET_YEOH_uy_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)



# BELLOW_MR PLOT

In [15]:
# %% RELOAD bundle and re-plot (save PDF + PNG to same subfolder)
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ---- Fonts (Times-like) + avoid Type 3) ----
mpl.rcParams.update({
    'font.family': ['Times New Roman', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'text.usetex': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

# ===== SETTINGS =====
PARENT   = "save_plots"
SUB      = "BELLOW_MR"
BUNDLE   = "BELLOW_MR_snap30.npz"   # change if you saved a different name
PNG_DPI  = 1440
# ====================

path = os.path.join(PARENT, SUB, BUNDLE)
if not os.path.exists(path):
    raise FileNotFoundError("Bundle not found: " + os.path.abspath(path))

Z = np.load(path, allow_pickle=True)

# unpack
SNAP   = int(Z["snap"])
x_ic   = Z["x_ic"];  y_ic = Z["y_ic"]
u_star = Z["u_star"]; v_star = Z["v_star"]
u_pred = Z["u_pred"]; v_pred = Z["v_pred"]
XLIM   = tuple(Z["xlim"])
CMAP   = str(Z["cmap"])

label_ux     = Z["label_ux"].item()
label_ux_hat = Z["label_ux_hat"].item()
label_uy     = Z["label_uy"].item()
label_uy_hat = Z["label_uy_hat"].item()

umin_u = float(Z["umin_u"]); umax_u = float(Z["umax_u"])
vmin_v = float(Z["vmin_v"]); vmax_v = float(Z["vmax_v"])

norm_u = Normalize(vmin=umin_u, vmax=umax_u)
norm_v = Normalize(vmin=vmin_v, vmax=vmax_v)

def plot_deformation(
    x, y, u, v, comp_label, norm, component='u',
    cmap='jet', xlim=None, textbox_xy=(0.035, 0.05),
    title=None, save_stub=None,
    # --- font controls ---
    cbar_label_fs=28, cbar_tick_fs=18, box_fs=21,
    label_fs=28, tick_fs=18, title_fs=20,
):
    x = x.ravel(); y = y.ravel()
    u = u.ravel(); v = v.ravel()

    # sign convention
    x_def = -x - u
    y_def =  y + v

    if component == 'u':
        cfield = u
        label_text = f"Max {comp_label}: {u.max():.2f}\nMin {comp_label}: {u.min():.2f}"
    else:
        cfield = v
        label_text = f"Max {comp_label}: {v.max():.2f}\nMin {comp_label}: {v.min():.2f}"

    fig, ax = plt.subplots(figsize=(10, 6), dpi=1440, constrained_layout=False)

    # --- FIX: keep a constant margin so big labels don't get cut ---
    # these fractions are figure-relative and identical across all plots
    fig.subplots_adjust(left=0.14, right=0.90, bottom=0.22, top=0.95)

    # plots
    ax.scatter(-x, y, color='grey', alpha=1, s=0.009)                       # undeformed
    ax.scatter(x_def, y_def, c=cfield, cmap=cmap, norm=norm, s=1, alpha=1)  # deformed

    # colorbar (attached to ax; margins above leave consistent space)
    cbar = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax,
                        fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=cbar_tick_fs)

    # Stats box
    ax.text(0.03, 0.03, label_text, transform=ax.transAxes,
            fontsize=box_fs, va='bottom',linespacing=0.95,
            bbox=dict(boxstyle='round,pad=0.18,rounding_size=0.06', facecolor='white', alpha=0.4))

    # labels / ticks / aspect
    ax.set_xlabel('X Coordinate', fontsize=label_fs, labelpad=8)  # <-- pad
    ax.set_ylabel('Y Coordinate', fontsize=label_fs, labelpad=6)  # <-- pad
    ax.tick_params(axis='both', labelsize=tick_fs)
    ax.set_aspect('equal', adjustable='datalim')  # consistent box; adjust limits

    if xlim is not None:
        ax.set_xlim(*xlim)
    if title:
        ax.set_title(title, fontsize=title_fs, pad=6)

    # Save without tight cropping
    if save_stub:
        base = os.path.join(PARENT, SUB, save_stub)
        fig.savefig(base + ".pdf")
        fig.savefig(base + ".png", dpi=PNG_DPI)

    plt.close(fig)


# Reproduce the two plots (u_y GT and Prediction). Add u_x if needed.

plot_deformation(x_ic, y_ic, u_star, v_star, comp_label=label_ux, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=None, save_stub=f"BELLOW_MR_ux_EXACT_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)
plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_ux_hat, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=None, save_stub=f"BELLOW_MR_ux_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

plot_deformation(
    x_ic, y_ic, u_star, v_star,
    comp_label=label_uy, norm=norm_v, component='v', cmap=CMAP,
    xlim=XLIM, title=None, save_stub=f"BELLOW_MR_uy_EXACT_snap{SNAP}_reload",
    cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_uy_hat, norm=norm_v,
                 component='v', cmap=CMAP, xlim=XLIM, title=None, save_stub=f"BELLOW_MR_uy_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

# BELLOW_YEOH PLOT

In [16]:
# %% RELOAD bundle and re-plot (save PDF + PNG to same subfolder)
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ---- Fonts (Times-like) + avoid Type 3) ----
mpl.rcParams.update({
    'font.family': ['Times New Roman', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'text.usetex': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

# ===== SETTINGS =====
PARENT   = "save_plots"
SUB      = "BELLOW_YEOH"
BUNDLE   = "BELLOW_YEOH_snap30.npz"   # change if you saved a different name
PNG_DPI  = 1440
# ====================

path = os.path.join(PARENT, SUB, BUNDLE)
if not os.path.exists(path):
    raise FileNotFoundError("Bundle not found: " + os.path.abspath(path))

Z = np.load(path, allow_pickle=True)

# unpack
SNAP   = int(Z["snap"])
x_ic   = Z["x_ic"];  y_ic = Z["y_ic"]
u_star = Z["u_star"]; v_star = Z["v_star"]
u_pred = Z["u_pred"]; v_pred = Z["v_pred"]
XLIM   = tuple(Z["xlim"])
CMAP   = str(Z["cmap"])

label_ux     = Z["label_ux"].item()
label_ux_hat = Z["label_ux_hat"].item()
label_uy     = Z["label_uy"].item()
label_uy_hat = Z["label_uy_hat"].item()

umin_u = float(Z["umin_u"]); umax_u = float(Z["umax_u"])
vmin_v = float(Z["vmin_v"]); vmax_v = float(Z["vmax_v"])

norm_u = Normalize(vmin=umin_u, vmax=umax_u)
norm_v = Normalize(vmin=vmin_v, vmax=vmax_v)

def plot_deformation(
    x, y, u, v, comp_label, norm, component='u',
    cmap='jet', xlim=None, textbox_xy=(0.035, 0.05),
    title=None, save_stub=None,
    # --- font controls ---
    cbar_label_fs=28, cbar_tick_fs=18, box_fs=21,
    label_fs=28, tick_fs=18, title_fs=20,
):
    x = x.ravel(); y = y.ravel()
    u = u.ravel(); v = v.ravel()

    # sign convention
    x_def = -x - u
    y_def =  y + v

    if component == 'u':
        cfield = u
        label_text = f"Max {comp_label}: {u.max():.2f}\nMin {comp_label}: {u.min():.2f}"
    else:
        cfield = v
        label_text = f"Max {comp_label}: {v.max():.2f}\nMin {comp_label}: {v.min():.2f}"

    fig, ax = plt.subplots(figsize=(10, 6), dpi=1440, constrained_layout=False)

    # --- FIX: keep a constant margin so big labels don't get cut ---
    # these fractions are figure-relative and identical across all plots
    fig.subplots_adjust(left=0.14, right=0.90, bottom=0.22, top=0.95)

    # plots
    ax.scatter(-x, y, color='grey', alpha=1, s=0.009)                       # undeformed
    ax.scatter(x_def, y_def, c=cfield, cmap=cmap, norm=norm, s=1, alpha=1)  # deformed

    # colorbar (attached to ax; margins above leave consistent space)
    cbar = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax,
                        fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=cbar_tick_fs)

    # Stats box
    ax.text(0.03, 0.03, label_text, transform=ax.transAxes,
            fontsize=box_fs, va='bottom',linespacing=0.95,
            bbox=dict(boxstyle='round,pad=0.18,rounding_size=0.06', facecolor='white', alpha=0.4))

    # labels / ticks / aspect
    ax.set_xlabel('X Coordinate', fontsize=label_fs, labelpad=8)  # <-- pad
    ax.set_ylabel('Y Coordinate', fontsize=label_fs, labelpad=6)  # <-- pad
    ax.tick_params(axis='both', labelsize=tick_fs)
    ax.set_aspect('equal', adjustable='datalim')  # consistent box; adjust limits

    if xlim is not None:
        ax.set_xlim(*xlim)
    if title:
        ax.set_title(title, fontsize=title_fs, pad=6)

    # Save without tight cropping
    if save_stub:
        base = os.path.join(PARENT, SUB, save_stub)
        fig.savefig(base + ".pdf")
        fig.savefig(base + ".png", dpi=PNG_DPI)

    plt.close(fig)


# Reproduce the two plots (u_y GT and Prediction). Add u_x if needed.

plot_deformation(x_ic, y_ic, u_star, v_star, comp_label=label_ux, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=None, save_stub=f"BELLOW_YEOH_ux_EXACT_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)
plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_ux_hat, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=None, save_stub=f"BELLOW_YEOH_ux_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

plot_deformation(
    x_ic, y_ic, u_star, v_star,
    comp_label=label_uy, norm=norm_v, component='v', cmap=CMAP,
    xlim=XLIM, title=None, save_stub=f"BELLOW_YEOH_uy_EXACT_snap{SNAP}_reload",
    cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_uy_hat, norm=norm_v,
                 component='v', cmap=CMAP, xlim=XLIM, title=None, save_stub=f"BELLOW_YEOH_uy_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)



# BENDING_JOINT_MR PLOT

In [1]:
# %% RELOAD bundle and re-plot (save PDF + PNG to same subfolder)
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ---- Fonts (Times-like) + avoid Type 3) ----
mpl.rcParams.update({
    'font.family': ['Times New Roman', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'text.usetex': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

# ===== SETTINGS =====
PARENT   = "save_plots"
SUB      = "BENDING_JOINT_MR"
BUNDLE   = "BENDING_JOINT_MR_snap45.npz"   # change if you saved a different name
PNG_DPI  = 1440
# ====================

path = os.path.join(PARENT, SUB, BUNDLE)
if not os.path.exists(path):
    raise FileNotFoundError("Bundle not found: " + os.path.abspath(path))

Z = np.load(path, allow_pickle=True)

# unpack
SNAP   = int(Z["snap"])
x_ic   = Z["x_ic"];  y_ic = Z["y_ic"]
u_star = Z["u_star"]; v_star = Z["v_star"]
u_pred = Z["u_pred"]; v_pred = Z["v_pred"]
XLIM   = tuple(Z["xlim"])
CMAP   = str(Z["cmap"])

label_ux     = Z["label_ux"].item()
label_ux_hat = Z["label_ux_hat"].item()
label_uy     = Z["label_uy"].item()
label_uy_hat = Z["label_uy_hat"].item()

umin_u = float(Z["umin_u"]); umax_u = float(Z["umax_u"])
vmin_v = float(Z["vmin_v"]); vmax_v = float(Z["vmax_v"])

norm_u = Normalize(vmin=umin_u, vmax=umax_u)
norm_v = Normalize(vmin=vmin_v, vmax=vmax_v)

def plot_deformation(
    x, y, u, v, comp_label, norm, component='u',
    cmap='jet', xlim=None, textbox_xy=(0.035, 0.05),
    title=None, save_stub=None,
    # --- font controls ---
    cbar_label_fs=28, cbar_tick_fs=18, box_fs=21,
    label_fs=28, tick_fs=18, title_fs=20,
):
    x = x.ravel(); y = y.ravel()
    u = u.ravel(); v = v.ravel()

    # sign convention
    x_def = -x - u
    y_def =  y + v

    if component == 'u':
        cfield = u
        label_text = f"Max {comp_label}: {u.max():.2f}\nMin {comp_label}: {u.min():.2f}"
    else:
        cfield = v
        label_text = f"Max {comp_label}: {v.max():.2f}\nMin {comp_label}: {v.min():.2f}"

    fig, ax = plt.subplots(figsize=(10, 6), dpi=1440, constrained_layout=False)

    # --- FIX: keep a constant margin so big labels don't get cut ---
    # these fractions are figure-relative and identical across all plots
    fig.subplots_adjust(left=0.14, right=0.90, bottom=0.22, top=0.95)

    # plots
    ax.scatter(-x, y, color='grey', alpha=1, s=0.009)                       # undeformed
    ax.scatter(x_def, y_def, c=cfield, cmap=cmap, norm=norm, s=1, alpha=1)  # deformed

    # colorbar (attached to ax; margins above leave consistent space)
    cbar = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax,
                        fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=cbar_tick_fs)

    # Stats box
    ax.text(0.03, 0.03, label_text, transform=ax.transAxes,
            fontsize=box_fs, va='bottom',linespacing=0.95,
            bbox=dict(boxstyle='round,pad=0.18,rounding_size=0.06', facecolor='white', alpha=0.4))

    # labels / ticks / aspect
    ax.set_xlabel('X Coordinate', fontsize=label_fs, labelpad=8)  # <-- pad
    ax.set_ylabel('Y Coordinate', fontsize=label_fs, labelpad=6)  # <-- pad
    ax.tick_params(axis='both', labelsize=tick_fs)
    ax.set_aspect('equal', adjustable='datalim')  # consistent box; adjust limits

    if xlim is not None:
        ax.set_xlim(*xlim)
    if title:
        ax.set_title(title, fontsize=title_fs, pad=6)

    # Save without tight cropping
    if save_stub:
        base = os.path.join(PARENT, SUB, save_stub)
        fig.savefig(base + ".pdf")
        fig.savefig(base + ".png", dpi=PNG_DPI)

    plt.close(fig)




# Reproduce the two plots (u_y GT and Prediction). Add u_x if needed.

plot_deformation(x_ic, y_ic, u_star, v_star, comp_label=label_ux, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=None, save_stub=f"BENDING_JOINT_MR_ux_EXACT_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)
plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_ux_hat, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=None, save_stub=f"BENDING_JOINT_MR_ux_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

plot_deformation(
    x_ic, y_ic, u_star, v_star,
    comp_label=label_uy, norm=norm_v, component='v', cmap=CMAP,
    xlim=XLIM, title=None, save_stub=f"BENDING_JOINT_MR_uy_EXACT_snap{SNAP}_reload",
    cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_uy_hat, norm=norm_v,
                 component='v', cmap=CMAP, xlim=XLIM, title=None, save_stub=f"BENDING_JOINT_MR_uy_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)



# BENDING_JOINT_YEOH PLOT

In [18]:
# %% RELOAD bundle and re-plot (save PDF + PNG to same subfolder)
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ---- Fonts (Times-like) + avoid Type 3) ----
mpl.rcParams.update({
    'font.family': ['Times New Roman', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'text.usetex': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

# ===== SETTINGS =====
PARENT   = "save_plots"
SUB      = "BENDING_JOINT_YEOH"
BUNDLE   = "BENDING_JOINT_YEOH_snap45.npz"   # change if you saved a different name
PNG_DPI  = 1440
# ====================

path = os.path.join(PARENT, SUB, BUNDLE)
if not os.path.exists(path):
    raise FileNotFoundError("Bundle not found: " + os.path.abspath(path))

Z = np.load(path, allow_pickle=True)

# unpack
SNAP   = int(Z["snap"])
x_ic   = Z["x_ic"];  y_ic = Z["y_ic"]
u_star = Z["u_star"]; v_star = Z["v_star"]
u_pred = Z["u_pred"]; v_pred = Z["v_pred"]
XLIM   = tuple(Z["xlim"])
CMAP   = str(Z["cmap"])

label_ux     = Z["label_ux"].item()
label_ux_hat = Z["label_ux_hat"].item()
label_uy     = Z["label_uy"].item()
label_uy_hat = Z["label_uy_hat"].item()

umin_u = float(Z["umin_u"]); umax_u = float(Z["umax_u"])
vmin_v = float(Z["vmin_v"]); vmax_v = float(Z["vmax_v"])

norm_u = Normalize(vmin=umin_u, vmax=umax_u)
norm_v = Normalize(vmin=vmin_v, vmax=vmax_v)

def plot_deformation(
    x, y, u, v, comp_label, norm, component='u',
    cmap='jet', xlim=None, textbox_xy=(0.035, 0.05),
    title=None, save_stub=None,
    # --- font controls ---
    cbar_label_fs=28, cbar_tick_fs=18, box_fs=21,
    label_fs=28, tick_fs=18, title_fs=20,
):
    x = x.ravel(); y = y.ravel()
    u = u.ravel(); v = v.ravel()

    # sign convention
    x_def = -x - u
    y_def =  y + v

    if component == 'u':
        cfield = u
        label_text = f"Max {comp_label}: {u.max():.2f}\nMin {comp_label}: {u.min():.2f}"
    else:
        cfield = v
        label_text = f"Max {comp_label}: {v.max():.2f}\nMin {comp_label}: {v.min():.2f}"

    fig, ax = plt.subplots(figsize=(10, 6), dpi=1440, constrained_layout=False)

    # --- FIX: keep a constant margin so big labels don't get cut ---
    # these fractions are figure-relative and identical across all plots
    fig.subplots_adjust(left=0.14, right=0.90, bottom=0.22, top=0.95)

    # plots
    ax.scatter(-x, y, color='grey', alpha=1, s=0.009)                       # undeformed
    ax.scatter(x_def, y_def, c=cfield, cmap=cmap, norm=norm, s=1, alpha=1)  # deformed

    # colorbar (attached to ax; margins above leave consistent space)
    cbar = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax,
                        fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=cbar_tick_fs)

    # Stats box
    ax.text(0.03, 0.03, label_text, transform=ax.transAxes,
            fontsize=box_fs, va='bottom',linespacing=0.95,
            bbox=dict(boxstyle='round,pad=0.18,rounding_size=0.06', facecolor='white', alpha=0.4))

    # labels / ticks / aspect
    ax.set_xlabel('X Coordinate', fontsize=label_fs, labelpad=8)  # <-- pad
    ax.set_ylabel('Y Coordinate', fontsize=label_fs, labelpad=6)  # <-- pad
    ax.tick_params(axis='both', labelsize=tick_fs)
    ax.set_aspect('equal', adjustable='datalim')  # consistent box; adjust limits

    if xlim is not None:
        ax.set_xlim(*xlim)
    if title:
        ax.set_title(title, fontsize=title_fs, pad=6)

    # Save without tight cropping
    if save_stub:
        base = os.path.join(PARENT, SUB, save_stub)
        fig.savefig(base + ".pdf")
        fig.savefig(base + ".png", dpi=PNG_DPI)

    plt.close(fig)


# Reproduce the two plots (u_y GT and Prediction). Add u_x if needed.

plot_deformation(x_ic, y_ic, u_star, v_star, comp_label=label_ux, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=None, save_stub=f"BENDING_JOINT_YEOH_ux_EXACT_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)
plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_ux_hat, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=None, save_stub=f"BENDING_JOINT_YEOH_ux_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

plot_deformation(
    x_ic, y_ic, u_star, v_star,
    comp_label=label_uy, norm=norm_v, component='v', cmap=CMAP,
    xlim=XLIM, title=None, save_stub=f"BENDING_JOINT_YEOH_uy_EXACT_snap{SNAP}_reload",
    cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_uy_hat, norm=norm_v,
                 component='v', cmap=CMAP, xlim=XLIM, title=None, save_stub=f"BENDING_JOINT_YEOH_uy_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)



# SPRING MR PLOT

In [1]:
# %% RELOAD bundle and re-plot (save PDF + PNG to same subfolder)
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ---- Fonts (Times-like) + avoid Type 3) ----
mpl.rcParams.update({
    'font.family': ['Times New Roman', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'text.usetex': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

# ===== SETTINGS =====
PARENT   = "save_plots"
SUB      = "SPRING_MR"
BUNDLE   = "SPRING_MR_snap4.npz"   # change if you saved a different name
PNG_DPI  = 1440
# ====================

path = os.path.join(PARENT, SUB, BUNDLE)
if not os.path.exists(path):
    raise FileNotFoundError("Bundle not found: " + os.path.abspath(path))

Z = np.load(path, allow_pickle=True)

# unpack
SNAP   = int(Z["snap"])
x_ic   = Z["x_ic"];  y_ic = Z["y_ic"]
u_star = Z["u_star"]; v_star = Z["v_star"]
u_pred = Z["u_pred"]; v_pred = Z["v_pred"]
XLIM = (-100.0, 20.0) 
CMAP   = str(Z["cmap"])

label_ux     = Z["label_ux"].item()
label_ux_hat = Z["label_ux_hat"].item()
label_uy     = Z["label_uy"].item()
label_uy_hat = Z["label_uy_hat"].item()

umin_u = float(Z["umin_u"]); umax_u = float(Z["umax_u"])
vmin_v = float(Z["vmin_v"]); vmax_v = float(Z["vmax_v"])

norm_u = Normalize(vmin=umin_u, vmax=umax_u)
norm_v = Normalize(vmin=vmin_v, vmax=vmax_v)

def plot_deformation(
    x, y, u, v, comp_label, norm, component='u',
    cmap='jet', xlim=None, textbox_xy=(0.035, 0.05),
    title=None, save_stub=None,
    # --- font controls ---
    cbar_label_fs=28, cbar_tick_fs=18, box_fs=21,
    label_fs=28, tick_fs=18, title_fs=20,
):
    x = x.ravel(); y = y.ravel()
    u = u.ravel(); v = v.ravel()

    # sign convention
    x_def = x + u
    y_def =  -y - v

    if component == 'u':
        cfield = u
        label_text = f"Max {comp_label}: {u.max():.2f}\nMin {comp_label}: {u.min():.2f}"
    else:
        cfield = v
        label_text = f"Max {comp_label}: {v.max():.2f}\nMin {comp_label}: {v.min():.2f}"

    fig, ax = plt.subplots(figsize=(10, 6), dpi=1440, constrained_layout=False)

    # --- FIX: keep a constant margin so big labels don't get cut ---
    # these fractions are figure-relative and identical across all plots
    fig.subplots_adjust(left=0.14, right=0.90, bottom=0.22, top=0.95)

    # plots
    ax.scatter(x, -y, color='grey', alpha=0.8, s=0.009)                       # undeformed
    ax.scatter(x_def, y_def, c=cfield, cmap=cmap, norm=norm, s=1, alpha=1)  # deformed

    # colorbar (attached to ax; margins above leave consistent space)
    cbar = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax,
                        fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=cbar_tick_fs)

    # Stats box
    ax.text(0.03, 0.03, label_text, transform=ax.transAxes,
            fontsize=box_fs, va='bottom',linespacing=0.95,
            bbox=dict(boxstyle='round,pad=0.18,rounding_size=0.06', facecolor='white', alpha=0.4))

    # labels / ticks / aspect
    ax.set_xlabel('X Coordinate', fontsize=label_fs, labelpad=8)  # <-- pad
    ax.set_ylabel('Y Coordinate', fontsize=label_fs, labelpad=6)  # <-- pad
    ax.tick_params(axis='both', labelsize=tick_fs)
    ax.set_aspect('equal', adjustable='box')  # consistent box; adjust limits

    if xlim is not None:
        ax.set_xlim(*xlim)
    if title:
        ax.set_title(title, fontsize=title_fs, pad=6)

    # Save without tight cropping
    if save_stub:
        base = os.path.join(PARENT, SUB, save_stub)
        fig.savefig(base + ".pdf")
        fig.savefig(base + ".png", dpi=PNG_DPI)

    plt.close(fig)




# Reproduce the two plots (u_y GT and Prediction). Add u_x if needed.

plot_deformation(x_ic, y_ic, u_star, v_star, comp_label=label_ux, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=None, save_stub=f"SPRING_MR_ux_EXACT_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)
plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_ux_hat, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=None, save_stub=f"SPRING_MR_ux_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

plot_deformation(
    x_ic, y_ic, u_star, v_star,
    comp_label=label_uy, norm=norm_v, component='v', cmap=CMAP,
    xlim=XLIM, title=None, save_stub=f"SPRING_MR_uy_EXACT_snap{SNAP}_reload",
    cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_uy_hat, norm=norm_v,
                 component='v', cmap=CMAP, xlim=XLIM, title=None, save_stub=f"SPRING_MR_uy_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)



# SPRING YEOH PLOT

In [20]:
# %% RELOAD bundle and re-plot (save PDF + PNG to same subfolder)
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ---- Fonts (Times-like) + avoid Type 3) ----
mpl.rcParams.update({
    'font.family': ['Times New Roman', 'STIXGeneral', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'text.usetex': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

# ===== SETTINGS =====
PARENT   = "save_plots"
SUB      = "SPRING_YEOH"
BUNDLE   = "SPRING_YEOH_snap4.npz"   # change if you saved a different name
PNG_DPI  = 1440
# ====================

path = os.path.join(PARENT, SUB, BUNDLE)
if not os.path.exists(path):
    raise FileNotFoundError("Bundle not found: " + os.path.abspath(path))

Z = np.load(path, allow_pickle=True)

# unpack
SNAP   = int(Z["snap"])
x_ic   = Z["x_ic"];  y_ic = Z["y_ic"]
u_star = Z["u_star"]; v_star = Z["v_star"]
u_pred = Z["u_pred"]; v_pred = Z["v_pred"]
XLIM   = (-100.0, 20.0)
CMAP   = str(Z["cmap"])

label_ux     = Z["label_ux"].item()
label_ux_hat = Z["label_ux_hat"].item()
label_uy     = Z["label_uy"].item()
label_uy_hat = Z["label_uy_hat"].item()

umin_u = float(Z["umin_u"]); umax_u = float(Z["umax_u"])
vmin_v = float(Z["vmin_v"]); vmax_v = float(Z["vmax_v"])

norm_u = Normalize(vmin=umin_u, vmax=umax_u)
norm_v = Normalize(vmin=vmin_v, vmax=vmax_v)

def plot_deformation(
    x, y, u, v, comp_label, norm, component='u',
    cmap='jet', xlim=None, textbox_xy=(0.035, 0.05),
    title=None, save_stub=None,
    # --- font controls ---
    cbar_label_fs=28, cbar_tick_fs=18, box_fs=21,
    label_fs=28, tick_fs=18, title_fs=20,
):
    x = x.ravel(); y = y.ravel()
    u = u.ravel(); v = v.ravel()

    # sign convention
    x_def = x + u
    y_def =  -y - v

    if component == 'u':
        cfield = u
        label_text = f"Max {comp_label}: {u.max():.2f}\nMin {comp_label}: {u.min():.2f}"
    else:
        cfield = v
        label_text = f"Max {comp_label}: {v.max():.2f}\nMin {comp_label}: {v.min():.2f}"

    fig, ax = plt.subplots(figsize=(10, 6), dpi=1440, constrained_layout=False)

    # --- FIX: keep a constant margin so big labels don't get cut ---
    # these fractions are figure-relative and identical across all plots
    fig.subplots_adjust(left=0.14, right=0.90, bottom=0.22, top=0.95)

    # plots
    ax.scatter(x, -y, color='grey', alpha=0.8, s=0.009)                       # undeformed
    ax.scatter(x_def, y_def, c=cfield, cmap=cmap, norm=norm, s=1, alpha=1)  # deformed

    # colorbar (attached to ax; margins above leave consistent space)
    cbar = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax,
                        fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=cbar_tick_fs)

    # Stats box
    ax.text(0.03, 0.03, label_text, transform=ax.transAxes,
            fontsize=box_fs, va='bottom',linespacing=0.95,
            bbox=dict(boxstyle='round,pad=0.18,rounding_size=0.06', facecolor='white', alpha=0.4))

    # labels / ticks / aspect
    ax.set_xlabel('X Coordinate', fontsize=label_fs, labelpad=8)  # <-- pad
    ax.set_ylabel('Y Coordinate', fontsize=label_fs, labelpad=6)  # <-- pad
    ax.tick_params(axis='both', labelsize=tick_fs)
    ax.set_aspect('equal', adjustable='box')  # consistent box; adjust limits

    if xlim is not None:
        ax.set_xlim(*xlim)
    if title:
        ax.set_title(title, fontsize=title_fs, pad=6)

    # Save without tight cropping
    if save_stub:
        base = os.path.join(PARENT, SUB, save_stub)
        fig.savefig(base + ".pdf")
        fig.savefig(base + ".png", dpi=PNG_DPI)

    plt.close(fig)


# Reproduce the two plots (u_y GT and Prediction). Add u_x if needed.

plot_deformation(x_ic, y_ic, u_star, v_star, comp_label=label_ux, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=None, save_stub=f"SPRING_YEOH_ux_EXACT_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)
plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_ux_hat, norm=norm_u, component='u',
                 cmap=CMAP, xlim=XLIM, title=None, save_stub=f"SPRING_YEOH_ux_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

plot_deformation(
    x_ic, y_ic, u_star, v_star,
    comp_label=label_uy, norm=norm_v, component='v', cmap=CMAP,
    xlim=XLIM, title=None, save_stub=f"SPRING_YEOH_uy_EXACT_snap{SNAP}_reload",
    cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

plot_deformation(x_ic, y_ic, u_pred, v_pred, comp_label=label_uy_hat, norm=norm_v,
                 component='v', cmap=CMAP, xlim=XLIM, title=None, save_stub=f"SPRING_YEOH_uy_PRED_snap{SNAP}_reload",
                 cbar_label_fs=30, cbar_tick_fs=30, box_fs=35,label_fs=35, tick_fs=30, title_fs=22)

